# The Double Hadamard Test

An entangled unitary operator can be decomposed using a space-like cut of the form:

$$\Gamma := {(c_i,  V_i \oplus W_i) : i \hspace{1mm} \epsilon \hspace{1mm} [m]}$$

For some positive integer, m & random variables, (i, j, g) such that:  

$$ {i, j \hspace{1mm} \epsilon \hspace{1mm} m \hspace{1mm} | \hspace{1mm} c_i, \hspace{1mm} m > 0}$$

and 

$${g \hspace{1mm} \epsilon \hspace{1mm} (0, 1)} $$

In [63]:
import cudaq
import numpy as np
import random

# Set up the CUDA-Q target for multiple QPUs if available
cudaq.set_target("nvidia", option="mqpu")
print(f"Number of QPUs available: {cudaq.get_target().num_qpus()}")

Number of QPUs available: 1


We substitute a controlled two-qubit pauli from each ancilla to both qubits, in-place of the unitary.

$$ V_i / V_j = \ket{0}\bra{0} \oplus V_i + \ket{1}\bra{1} \oplus V_j$$

In [64]:
def double_hadamard_test(circuit):
    num_qubits = len(circuit)
    index_to_cut = 3 # top qubit index
    sublength = num_qubits - index_to_cut
    theta = np.pi # rotation angle

    # Randomly generate i=k, j=l, & g
    k = random.randint(0, num_qubits)
    l = random.randint(0, num_qubits)
    g = random.randint(0, 1) # g = {0, 1}

    # pauli dictionary
    pauli_mapping = {
        "I" : 0, "X": 1, "Y": 2, "Z": 3
    }

    # Register custom unitary operations (all gates except the one being cut)
    base_name = 'U'
    for i in range(len(circuit)):
        if i != index_to_cut:  # Skip the gate at the cut index
            cudaq.register_operation(f'{base_name}_{i}', circuit[i])

    # Build 1st subcircuit
    def build_kernel_1(gate_id):
        # Defining kernel using the builder function
        kernel = cudaq.make_kernel() 
        q1 = kernel.qalloc(index_to_cut + 2) # +1 for zero-index, +1 for ancilla
        
        kernel.h(q1[0])
        if g != 1:
            kernel.s(q1[0])

        # Apply the custom operations to the first subcircuit
        for i in range(1, index_to_cut):
            kernel.__getattr__(f'{base_name}_{i}')(q1[i], q1[i+1])

        # Apply the specific gate based on the ID passed in
        # I is index 0 - do nothing
        if gate_id == 1:
            kernel.crx(theta, q1[0], q1[index_to_cut])
        elif gate_id == 2:
            kernel.cry(theta, q1[0], q1[index_to_cut])
        elif gate_id == 3:
            kernel.crz(theta, q1[0], q1[index_to_cut])
        
        kernel.h(q1[0])
        kernel.mz(q1)
        return kernel

    # Build 2nd subcircuit
    def build_kernel_2(gate_id):
        kernel = cudaq.make_kernel()
        q2 = kernel.qalloc(sublength + 1)
        
        kernel.h(q2[sublength])
        if g != 1:
            kernel.s(q2[sublength])

        # Apply specific gate based on ID passed in
        if gate_id == 1:
            kernel.crx(theta, q2[sublength], q2[0])
        elif gate_id == 2:
            kernel.cry(theta, q2[sublength], q2[0])
        elif gate_id == 3:
            kernel.crz(theta, q2[sublength], q2[0])
            
        # Apply the remaining custom operations to the second subcircuit
        for i in range(index_to_cut+1, len(circuit)):
            j = i - (index_to_cut + 1) # offset the index to start at 0
            kernel.__getattr__(f'{base_name}_{i}')(q2[j], q2[j+1])
            
        kernel.h(q2[sublength])
        kernel.mz(q2)
        return kernel

    results1 = []
    results2 = []
    k2 = []
    # --- Iterate through all pauli gates and apply to each subcircuit ---
    idx = 0
    for name1, id2 in pauli_mapping.items():
        # Build kernel 2
        k2.append(build_kernel_2(id2))
        results2.append([cudaq.sample(k2[idx]), name1])
        idx = idx + 1

    for name1, id1 in pauli_mapping.items():
        # Build kernel 1
        k1 = build_kernel_1(id1)
        results1.append([cudaq.sample(k1), name1])
        
        idx = 0
        for name2, id2 in pauli_mapping.items():  
            # Test each combination 
            print(f"Testing combination: {name1} - {name2}")
            print(cudaq.draw(k1))
            print(cudaq.draw(k2[idx]))
            idx = idx + 1
            
            print("===================================")
    return results1, results2 # return both results lists


# Double Hadamard Test File

Observe a base circuit and pass it to the double hadamard test
 - Compute expectation value of circuit
 - Consider Ising models and other observable custom unitaries

 M = (.12) XY + (.34) YZ

In [65]:
cnot_gate = np.array([[1, 0, 0, 0],[0, 1, 0, 0],[0, 0, 0, 1],[0, 0, 1, 0]])
circuit = [cnot_gate, cnot_gate, cnot_gate, cnot_gate, cnot_gate]  # 5 CNOT gates

base_name = 'U'
for i in range(len(circuit)):
    cudaq.register_operation(f'{base_name}_{i}', circuit[i])

# Construct the base kernel
kernel = cudaq.make_kernel()
q = kernel.qalloc(len(circuit) + 1)

for i in range(len(circuit)):
    kernel.__getattr__(f'{base_name}_{i}')(q[i], q[i+1])
    
base_results = cudaq.sample(kernel)

results1, results2 = double_hadamard_test(circuit)

Testing combination: I - I
      ╭───╮  ╭───╮ 
q0 : ─┤ h ├──┤ h ├─
     ╭┴───┴╮ ╰───╯ 
q1 : ┤     ├───────
     │ U_1 │╭─────╮
q2 : ┤     ├┤     ├
     ╰─────╯│ U_2 │
q3 : ───────┤     ├
            ╰─────╯

     ╭─────╮     
q0 : ┤     ├─────
     │ U_4 │     
q1 : ┤     ├─────
     ╰┬───┬╯╭───╮
q2 : ─┤ h ├─┤ h ├
      ╰───╯ ╰───╯

Testing combination: I - X
      ╭───╮  ╭───╮ 
q0 : ─┤ h ├──┤ h ├─
     ╭┴───┴╮ ╰───╯ 
q1 : ┤     ├───────
     │ U_1 │╭─────╮
q2 : ┤     ├┤     ├
     ╰─────╯│ U_2 │
q3 : ───────┤     ├
            ╰─────╯

          ╭───────────╮╭─────╮
q0 : ─────┤ rx(3.142) ├┤     ├
          ╰─────┬─────╯│ U_4 │
q1 : ───────────┼──────┤     ├
     ╭───╮      │      ╰┬───┬╯
q2 : ┤ h ├──────●───────┤ h ├─
     ╰───╯              ╰───╯ 

Testing combination: I - Y
      ╭───╮  ╭───╮ 
q0 : ─┤ h ├──┤ h ├─
     ╭┴───┴╮ ╰───╯ 
q1 : ┤     ├───────
     │ U_1 │╭─────╮
q2 : ┤     ├┤     ├
     ╰─────╯│ U_2 │
q3 : ───────┤     ├
            ╰─────╯

          ╭───────────╮╭─────╮


In [66]:
# Print results
print(f"Original Circuit expectation value: {base_results.expectation():.4f}")
print(cudaq.draw(kernel))

for res1, name1 in results1:
    for res2, name2 in results2:
        print(f"Combination: {name1} - {name2}")
        print(f" Subcircuit 1 expectation value: {res1.expectation():.4f}")
        print(f" Subcircuit 2 expectation value: {res2.expectation():.4f}")

Original Circuit expectation value: 1.0000
     ╭─────╮                            
q0 : ┤     ├────────────────────────────
     │ U_0 │╭─────╮                     
q1 : ┤     ├┤     ├─────────────────────
     ╰─────╯│ U_1 │╭─────╮              
q2 : ───────┤     ├┤     ├──────────────
            ╰─────╯│ U_2 │╭─────╮       
q3 : ──────────────┤     ├┤     ├───────
                   ╰─────╯│ U_3 │╭─────╮
q4 : ─────────────────────┤     ├┤     ├
                          ╰─────╯│ U_4 │
q5 : ────────────────────────────┤     ├
                                 ╰─────╯

Combination: I - I
 Subcircuit 1 expectation value: 1.0000
 Subcircuit 2 expectation value: 1.0000
Combination: I - X
 Subcircuit 1 expectation value: 1.0000
 Subcircuit 2 expectation value: 0.0420
Combination: I - Y
 Subcircuit 1 expectation value: 1.0000
 Subcircuit 2 expectation value: 0.0060
Combination: I - Z
 Subcircuit 1 expectation value: 1.0000
 Subcircuit 2 expectation value: -0.0120
Combination: X - I
 Subcir